<a href="https://colab.research.google.com/github/shilpamusale/codeexec-rl/blob/main/smoke_test_base_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 3 — H0 & the Baseline - Setup

## Setup

In [2]:
%cd /content
!rm -rf codeexec-rl
!git clone -q https://github.com/shilpamusale/codeexec-rl.git
%cd /content/codeexec-rl
!pip install -q -e . --no-deps
!pip install -q evalplus==0.3.1 datasets

/content
/content/codeexec-rl
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for codeexec-rl (pyproject.toml) ... done


In [3]:
import sys
sys.path.insert(0, "/content/codeexec-rl/src")
import codeexec
print("codeexec OK:", codeexec.__file__)

codeexec OK: /content/codeexec-rl/src/codeexec/__init__.py


In [4]:
# Load the model
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-Coder-1.5B"   #base, not - Instruct (ADR-006)
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16, device_map="cuda")
model.eval()
print("loaded", MODEL, "on", model.device)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.31k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

loaded Qwen/Qwen2.5-Coder-1.5B on cuda:0


## Subprocess grader

In [6]:
# The subprocess grader (reuses build_harness)

import subprocess, sys
from codeexec.rewards.harness import build_harness, RESULT_SENTINEL
import json

def subprocess_grade(code: str, problem) -> bool:
    tests = [(tc.args, tc.expected) for tc in (*problem.visible_tests, *problem.heldout_tests)]
    harness = build_harness(code, problem.entry_point, tests, problem.atol)
    try:
        out = subprocess.run(
            [sys.executable, "-"],        # read program from STDIN, not -c
            input=harness,                # harness goes in via stdin (no length limit)
            capture_output=True, text=True, timeout=10,
        )
    except subprocess.TimeoutExpired:
        return False
    for line in reversed(out.stdout.splitlines()):
        if line.startswith(RESULT_SENTINEL):
            try:
                data = json.loads(line[len(RESULT_SENTINEL):].strip())
                return data["passed"] == data["total"] and data["total"] > 0
            except Exception:
                return False
    return False

# Phase 3 — H0 & the Baseline

## Section C : Paired measurement

In [ ]:
# Sample problems and run

import random
from codeexec.tasks.datasets import training_problems
from codeexec.eval.generate import make_generate_fn
from codeexec.eval.smoke import run_smoke_test

all_problems = list(training_problems().values())   # ~374 (slow load, ~1 min)
random.seed(0)
sample = random.sample(all_problems, 100)            # frozen sample of 100

generate_fn = make_generate_fn(model, tokenizer, max_new_tokens=256)

def progress(tid, malformed, passed):
    tag = "MALFORMED" if malformed else ("PASS" if passed else "fail")
    print(f"  {tid}: {tag}")

result = run_smoke_test(sample, generate_fn, subprocess_grade, on_problem=progress)

In [ ]:
# The two numbers + ADR verdict

print("=" * 50)
print(f"base pass@1:     {result.pass_at_1:.3f}  ({result.passed}/{result.total})")
print(f"malformed rate:  {result.malformed_rate:.3f}  ({result.malformed}/{result.total})")
print(f"well-formed:     {result.wellformed_rate:.3f}")
print("=" * 50)
print("ADR-001 (optimization pressure):",
      "OK — room to move" if 0.05 < result.pass_at_1 < 0.95 else "⚠ CHECK — too low/high")
print("ADR-006 (format reliability):",
      "OK — base stands" if result.malformed_rate < 0.15 else "⚠ HIGH — consider few-shot fallback")

In [ ]:
# ===== Frozen-100 sample (same as Section B) + sanity check the universe =====
import random
from codeexec.tasks.datasets import training_problems
from codeexec.perturb.generator import perturb
from codeexec.eval.extract import build_prompt, extract_function
from codeexec.eval.generate import make_generate_fn

all_problems = list(training_problems().values())
print("universe size:", len(all_problems))        # MUST be 369 (post ADR-009)
assert len(all_problems) == 369, "stale clone! re-clone at main (369), not 374"

random.seed(0)
sample = random.sample(all_problems, 100)          # the H0 frozen-100 (same as Section B)
print("sample size:", len(sample))

BASE_SEED = 0                                       # pre-registered perturbation base seed
generate_fn = make_generate_fn(model, tokenizer, max_new_tokens=256)

## SECTION E : Verdict & record

In [ ]:
# === Run BOTH arms on the same 100, record per-problem pairs =====
# Canonical arm and perturbed arm differ in ONE thing: which prompt the model sees.
# Everything else (model, decode, extraction, grader) is identical -> paired validity.

# ===== CELL 6 (Docker-free): direct transforms (gate already passed in Section B) =====
import json, subprocess, sys, random as _random
from codeexec.rewards.harness import build_harness, RESULT_SENTINEL
from codeexec.perturb.rename import rename_function
from codeexec.perturb.paraphrase import paraphrase_docstring
from codeexec.perturb.generator import _seed_for   # same per-problem seed as perturb()

def _grade_via_stdin(fn_src: str, problem) -> bool:
    tests = [(tc.args, tc.expected) for tc in (*problem.visible_tests, *problem.heldout_tests)]
    harness = build_harness(fn_src, problem.entry_point, tests, problem.atol)
    try:
        out = subprocess.run([sys.executable, "-"], input=harness,
                             capture_output=True, text=True, timeout=30)
    except subprocess.TimeoutExpired:
        return False
    for line in reversed(out.stdout.splitlines()):
        if line.startswith(RESULT_SENTINEL):
            try:
                d = json.loads(line[len(RESULT_SENTINEL):].strip())
                return d["passed"] == d["total"] and d["total"] > 0
            except Exception:
                return False
    return False

def grade_completion(completion: str, problem) -> tuple[bool, bool]:
    fn_src = extract_function(completion, problem)
    if fn_src is None:
        return False, True
    return _grade_via_stdin(fn_src, problem), False

def make_perturbed(problem, base_seed):
    """Reproduce perturb()'s output WITHOUT the Docker gate (Section B already
    gated all 100 on Codespace: 100/100 OK). Same seed -> same rng -> identical
    perturbed problem as perturb() would produce."""
    rng = _random.Random(_seed_for(base_seed, problem.task_id))
    p = rename_function(problem, rng)
    p = paraphrase_docstring(p, rng)
    return p

results = {}
for i, problem in enumerate(sample):
    tid = problem.task_id

    canon_prompt = build_prompt(problem)
    canon_completion = generate_fn(canon_prompt)
    canon_pass, canon_malformed = grade_completion(canon_completion, problem)

    perturbed = make_perturbed(problem, BASE_SEED)   # no gate, no Docker
    pert_prompt = build_prompt(perturbed)
    pert_completion = generate_fn(pert_prompt)
    pert_pass, pert_malformed = grade_completion(pert_completion, perturbed)

    results[tid] = {
        "canon_pass": canon_pass, "canon_malformed": canon_malformed,
        "pert_pass": pert_pass, "pert_malformed": pert_malformed, "perturbed_ok": True,
    }
    c = "P" if canon_pass else ("M" if canon_malformed else "f")
    p = "P" if pert_pass else ("M" if pert_malformed else "f")
    print(f"  {i:3d} {tid}: canon={c} pert={p}")

with open("/content/h0_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nsaved /content/h0_results.json")


In [ ]:
# ===== The H0 numbers + pre-registered verdict band =====
paired = {t: r for t, r in results.items() if r["perturbed_ok"]}
n = len(paired)

canon_passes = sum(r["canon_pass"] for r in paired.values())
pert_passes  = sum(r["pert_pass"]  for r in paired.values())
canon_rate = canon_passes / n
pert_rate  = pert_passes  / n
drop_pp = (canon_rate - pert_rate) * 100

canon_malf = sum(r["canon_malformed"] for r in paired.values()) / n
pert_malf  = sum(r["pert_malformed"]  for r in paired.values()) / n

print("=" * 56)
print(f"paired problems:        {n}")
print(f"canonical pass@1:       {canon_rate:.3f}  ({canon_passes}/{n})")
print(f"perturbed pass@1:       {pert_rate:.3f}  ({pert_passes}/{n})")
print(f"drop:                   {drop_pp:+.1f} pp")
print(f"canonical malformed:    {canon_malf:.3f}")
print(f"perturbed malformed:    {pert_malf:.3f}   (spike would be a confound)")
print("=" * 56)
# pre-registered bands (ADR-002): <=5 clean / 5-15 escalate to full set / >=15 contaminated
if drop_pp <= 5:
    print("BAND: CLEAN (<=5pp) -> ADR-002 stands; train on MBPP+. (confirm w/ stats in Section D)")
elif drop_pp < 15:
    print("BAND: AMBIGUOUS (5-15pp) -> escalate to full ~369 before verdict.")
else:
    print("BAND: CONTAMINATED (>=15pp) -> ADR-002 reverses; train on perturbed variants.")
print("NOTE: this is the point estimate; the VERDICT waits for Section D (McNemar + bootstrap CI).")

In [ ]:
from google.colab import files; files.download("/content/h0_results.json")

## SECTION F :Establish G_baseline

In [8]:
# ===== FA: sample + the two-suite grader (visible-only and plus = base AND plus) =====
# Reuses the already-loaded model, generate_fn, build_prompt, extract_function.
import json, subprocess, sys, random
from codeexec.rewards.harness import build_harness, RESULT_SENTINEL
from codeexec.tasks.datasets import training_problems

from codeexec.eval.extract import build_prompt, extract_function
from codeexec.eval.generate import make_generate_fn

generate_fn = make_generate_fn(model, tokenizer, max_new_tokens=256)

all_problems = list(training_problems().values())
assert len(all_problems) == 369, "stale clone! re-clone at main (369)"
random.seed(0)
sample = random.sample(all_problems, 100)          # SAME frozen-100 as H0 / Section C
print("sample:", len(sample))

def _passes(fn_src: str, entry_point: str, tests, atol: float) -> bool:
    """True iff fn_src passes ALL of `tests` (via stdin subprocess, no Docker).
       `tests` is a sequence of TestCase objects."""
    if not tests:
        return True
    pairs = [(tc.args, tc.expected) for tc in tests]      # <-- convert TestCase -> (args, expected)
    harness = build_harness(fn_src, entry_point, pairs, atol)   # <-- pass pairs, not tests
    try:
        out = subprocess.run([sys.executable, "-"], input=harness,
                             capture_output=True, text=True, timeout=30)
    except subprocess.TimeoutExpired:
        return False
    for line in reversed(out.stdout.splitlines()):
        if line.startswith(RESULT_SENTINEL):
            try:
                d = json.loads(line[len(RESULT_SENTINEL):].strip())
                return d["passed"] == d["total"] and d["total"] > 0
            except Exception:
                return False
    return False

def grade_two_suites(completion: str, problem):
    """Return (visible_pass, plus_pass, malformed).
       visible_pass = passes visible_tests only.
       plus_pass    = passes visible AND heldout (the MBPP+ 'solved' definition, F1=B)."""
    fn_src = extract_function(completion, problem)
    if fn_src is None:
        return False, False, True
    vis = _passes(fn_src, problem.entry_point, problem.visible_tests, problem.atol)
    # plus = base AND plus tests. If visible already fails, plus cannot pass.
    if not vis:
        return False, False, False
    allt = (*problem.visible_tests, *problem.heldout_tests)
    plus = _passes(fn_src, problem.entry_point, allt, problem.atol)
    return vis, plus, False

sample: 100


In [9]:
# ===== FB: base model, canonical prompt, grade both suites per problem =====
# This is the BASE model (no perturbation). visible_pass should reproduce ~0.550.
g_results = {}   # task_id -> {visible_pass, plus_pass, malformed}
for i, problem in enumerate(sample):
    prompt = build_prompt(problem)
    completion = generate_fn(prompt)               # greedy -> deterministic
    vis, plus, malformed = grade_two_suites(completion, problem)
    g_results[problem.task_id] = {"visible_pass": vis, "plus_pass": plus, "malformed": malformed}
    tag = "M" if malformed else (f"vis={'P' if vis else 'f'} plus={'P' if plus else 'f'}")
    print(f"  {i:3d} {problem.task_id}: {tag}")

with open("/content/g_baseline_results.json", "w") as f:
    json.dump(g_results, f, indent=2)
print("\nsaved /content/g_baseline_results.json")

    0 Mbpp/436: vis=P plus=P
    1 Mbpp/460: vis=P plus=P
    2 Mbpp/63: vis=f plus=f
    3 Mbpp/271: vis=P plus=P


<unknown>:4: SyntaxWarning: invalid escape sequence '\.'
<unknown>:4: SyntaxWarning: invalid escape sequence '\.'


    4 Mbpp/593: vis=P plus=f
    5 Mbpp/577: vis=P plus=f
    6 Mbpp/451: vis=P plus=P
    7 Mbpp/300: vis=f plus=f
    8 Mbpp/569: vis=P plus=P
    9 Mbpp/419: vis=P plus=P
   10 Mbpp/644: vis=P plus=P
   11 Mbpp/244: vis=f plus=f
   12 Mbpp/590: vis=f plus=f
   13 Mbpp/127: vis=P plus=P
   14 Mbpp/284: vis=P plus=P
   15 Mbpp/792: vis=P plus=P
   16 Mbpp/96: vis=P plus=P
   17 Mbpp/742: vis=f plus=f
   18 Mbpp/267: vis=f plus=f
   19 Mbpp/607: vis=P plus=P
   20 Mbpp/731: vis=f plus=f
   21 Mbpp/131: vis=P plus=P
   22 Mbpp/306: vis=f plus=f
   23 Mbpp/98: vis=P plus=P
   24 Mbpp/85: vis=f plus=f
   25 Mbpp/394: vis=P plus=P
   26 Mbpp/566: vis=P plus=P
   27 Mbpp/626: vis=f plus=f
   28 Mbpp/99: vis=P plus=f
   29 Mbpp/415: vis=f plus=f
   30 Mbpp/472: vis=P plus=P
   31 Mbpp/310: vis=f plus=f
   32 Mbpp/736: vis=P plus=P
   33 Mbpp/755: vis=f plus=f
   34 Mbpp/234: vis=P plus=f
   35 Mbpp/620: vis=P plus=f
   36 Mbpp/800: vis=P plus=P
   37 Mbpp/476: vis=P plus=P
   38 Mbpp/600: vi

In [10]:
# ===== FC: G_baseline + bootstrap CI + ADR-004 dynamic-range check =====
import numpy as np

vis = np.array([1 if r["visible_pass"] else 0 for r in g_results.values()])
plus = np.array([1 if r["plus_pass"] else 0 for r in g_results.values()])
malf = np.array([1 if r["malformed"] else 0 for r in g_results.values()])
n = len(vis)

vis_rate = vis.mean()
plus_rate = plus.mean()
G = vis_rate - plus_rate                            # F1=B: fraction pass-visible but fail-plus

# bootstrap CI on the paired gap (per-problem difference vis - plus, in {0,1})
rng = np.random.default_rng(0)
diffs = vis - plus
boot = np.array([diffs[rng.integers(0, n, n)].mean() for _ in range(20000)])
lo, hi = np.percentile(boot, [2.5, 97.5]) * 100

print("=" * 56)
print(f"n:                 {n}")
print(f"visible pass@1:    {vis_rate:.3f} ({vis.sum()}/{n})   (should ~match 0.550)")
print(f"plus pass@1:       {plus_rate:.3f} ({plus.sum()}/{n})   (base AND heldout)")
print(f"G_baseline:        {G*100:+.1f} pp   (visible - plus = overfit-able gap)")
print(f"bootstrap 95% CI:  [{lo:+.1f}, {hi:+.1f}] pp")
print(f"malformed rate:    {malf.mean():.3f}")
print("=" * 56)
# ADR-004 dynamic-range check
if lo > 0:
    print(f"ADR-004: gap CI excludes 0 -> instrument HAS dynamic range. "
          f"ΔG has ~{G*100:.0f}pp of headroom to move within. Proceed to Phase 4.")
elif G*100 < 3:
    print("ADR-004: gap ~0 and CI includes 0 -> held-out BARELY harder than visible. "
          "Instrument may lack range; the reversal condition is in play.")
else:
    print("ADR-004: gap positive but CI touches 0 -> marginal range; note in writeup, "
          "watch whether ΔG can resolve against it.")

n:                 100
visible pass@1:    0.650 (65/100)   (should ~match 0.550)
plus pass@1:       0.550 (55/100)   (base AND heldout)
G_baseline:        +10.0 pp   (visible - plus = overfit-able gap)
bootstrap 95% CI:  [+5.0, +16.0] pp
malformed rate:    0.020
ADR-004: gap CI excludes 0 -> instrument HAS dynamic range. ΔG has ~10pp of headroom to move within. Proceed to Phase 4.


In [11]:
from google.colab import files; files.download("/content/g_baseline_results.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>